# UpgradePvP - Association Rules Notebook

This notebook builds association rules from the **UpgradePvP** dataset.

It includes:

- **Apriori**
- **AprioriTID**
- **ECLAT**
- **FP-Growth**
- **multidimensional associations** built from purchases + kill/death features + win proxy
- the main interestingness measures explicitly used in the slides: **support**, **confidence**, **lift**

Data sources:

- `upgradepvp_20250218-sqlite.sql`

## Notes about the dataset

- Players are displayed by **name**.
- A `DIAMOND` purchase is treated as a **win proxy**.
- For market-basket style mining, repeated purchases inside the same player-game are reduced to a **set** of items.


In [ ]:
%pip install -q mlxtend nbformat

In [ ]:
import math
import re
import sqlite3
import time
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth
from mlxtend.preprocessing import TransactionEncoder

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 200)

## 1. Paths and SQLite database

The task said the SQL dump was already loaded into a SQLite database. To make the notebook portable, the next cell recreates `upgradepvp.db` **only if it does not exist yet**.


In [ ]:
BASE = Path('.')
SQL_PATH = BASE / 'upgradepvp_20250218-sqlite.sql'
DB_PATH = BASE / 'upgradepvp.db'


def ensure_sqlite_db(sql_path: Path, db_path: Path) -> None:
    if db_path.exists():
        return
    conn = sqlite3.connect(db_path)
    try:
        conn.executescript(sql_path.read_text(encoding='utf-8'))
        conn.commit()
    finally:
        conn.close()


ensure_sqlite_db(SQL_PATH, DB_PATH)
DB_PATH.resolve()

## 2. Load tables

In [ ]:
def load_tables(db_path: Path):
    conn = sqlite3.connect(db_path)
    try:
        purchase = pd.read_sql(
            '''
            SELECT pu.*, pl.player_name
            FROM purchase pu
            JOIN player pl ON pu.player_uuid = pl.player_uuid
            ''',
            conn,
        )
        death = pd.read_sql(
            '''
            SELECT d.*, pm.player_name AS murderer_name, pd.player_name AS dead_name
            FROM death d
            JOIN player pm ON d.murderer_uuid = pm.player_uuid
            JOIN player pd ON d.dead_uuid = pd.player_uuid
            ''',
            conn,
        )
        game_player = pd.read_sql(
            '''
            SELECT DISTINCT gp.game_id, gp.player_uuid, pl.player_name
            FROM game_player gp
            JOIN player pl ON gp.player_uuid = pl.player_uuid
            ''',
            conn,
        )
        player = pd.read_sql('SELECT * FROM player', conn)
    finally:
        conn.close()
    return purchase, death, game_player, player


purchase_raw, death_raw, game_player, player = load_tables(DB_PATH)

print('purchase rows:', len(purchase_raw))
print('death rows:', len(death_raw))
print('distinct players:', len(player))
print('games:', game_player['game_id'].nunique())

## 3. Parse purchases

We split `purchase.item` into:

- `item_base`
- `item_detail`

Examples:

- `WOOD_SWORD$null` -> base item only
- `ENCHANTED_BOOK$§bFlame` -> base `ENCHANTED_BOOK`, detail `Flame`

We also classify items into **tiers / categories**.


In [ ]:
def normalize_detail(detail):
    if detail is None or pd.isna(detail):
        return None
    detail = str(detail)
    if detail.lower() == 'null':
        return None
    detail = re.sub(r'^§.', '', detail).strip()
    mapping = {
        'Feather Falling': 'FallProt',
        'Fire Aspect': 'FireAspect',
        'Fire Protection': 'FireProt',
        'Projectile Protection': 'ProjProt',
        'Blast Protection': 'BlastProt',
        'Aqua Affinity': 'AquaInf',
    }
    return mapping.get(detail, detail)


def classify_item(base, detail):
    if base.endswith('_SWORD'):
        return 'Sword', base.replace('_SWORD', '').title(), base

    armor_match = re.match(r'^(LEATHER|CHAINMAIL|GOLD|IRON|DIAMOND)_(HELMET|CHESTPLATE|LEGGINGS|BOOTS)$', base)
    if armor_match:
        material, part = armor_match.groups()
        material = {'CHAINMAIL': 'Chain'}.get(material, material.title())
        return 'Armor', material, part.title()

    if base == 'ENCHANTED_BOOK':
        return 'Enchant', detail, detail

    other_map = {
        'BOW': ('Other', 'Bow', 'Bow'),
        'ARROW': ('Other', 'Arrow', 'Arrow'),
        'FISHING_ROD': ('Other', 'FishingRod', 'FishingRod'),
        'SNOW_BALL': ('Other', 'SnowBall', 'SnowBall'),
        'KEEP_INVENTORY': ('Other', 'KeepInventory', 'KeepInventory'),
        'DIAMOND': ('Outcome', 'Win', 'DiamondWinProxy'),
    }
    return other_map.get(base, ('Other', base.title(), base.title()))


def canonical_purchase_token(row):
    if row['main_category'] == 'Enchant':
        return f"ENCHANT:{row['item_detail']}"
    return row['item_base']


purchase = purchase_raw.copy()
purchase['item_base'] = purchase['item'].str.split('$').str[0]
purchase['item_detail_raw'] = purchase['item'].str.split('$').str[1]
purchase['item_detail'] = purchase['item_detail_raw'].map(normalize_detail)

classified = purchase.apply(
    lambda row: classify_item(row['item_base'], row['item_detail']),
    axis=1,
    result_type='expand'
)
classified.columns = ['main_category', 'sub_category', 'item_kind']

purchase = pd.concat([purchase, classified], axis=1)
purchase['canonical_token'] = purchase.apply(canonical_purchase_token, axis=1)

purchase[['item', 'item_base', 'item_detail', 'main_category', 'sub_category', 'canonical_token']].drop_duplicates().sort_values('item')

## 4. Create purchase transactions

A transaction is defined here as **one player in one game**.

For market-basket style association rules we use **set semantics**:
if a player bought the same thing many times in the same game, the transaction still contains the item only once.


In [63]:
# 1. Prepare purchase events
purchases_for_tx = purchase[['game_id', 'player_uuid', 'player_name', 'created_at', 'canonical_token', 'item_base']].copy()
purchases_for_tx['event_type'] = 'purchase'

# 2. Prepare death events (from the perspective of the victim)
deaths_for_tx = death_raw[['game_id', 'dead_uuid', 'created_at', 'dead_name']].copy()
deaths_for_tx = deaths_for_tx.rename(columns={'dead_uuid': 'player_uuid', 'dead_name': 'player_name'})
deaths_for_tx['event_type'] = 'death'
deaths_for_tx['canonical_token'] = None
deaths_for_tx['item_base'] = None

# 3. Combine into a single timeline and sort chronologically
# Sorting by 'event_type' ensures that if a purchase and death happen in the exact same second,
# 'purchase' is processed before 'death' (alphabetical).
events = pd.concat([purchases_for_tx, deaths_for_tx], ignore_index=True)
events['created_at'] = pd.to_datetime(events['created_at'])
events = events.sort_values(['game_id', 'player_uuid', 'created_at', 'event_type'])

# 4. Iterate through the event stream to build "life" transactions
transactions_list = []

for (game_id, player_uuid, player_name), group in events.groupby(['game_id', 'player_uuid', 'player_name']):
    current_basket = set()
    keep_inventory = False
    
    for _, row in group.iterrows():
        if row['event_type'] == 'purchase':
            # Ignore NaNs just in case
            if pd.notna(row['canonical_token']):
                current_basket.add(row['canonical_token'])
            
            # Check if they unlocked the permanent keep inventory perk
            if row['item_base'] == 'KEEP_INVENTORY':
                keep_inventory = True
                
        elif row['event_type'] == 'death':
            # If they die without the perk, the current basket is finalized
            if not keep_inventory:
                if current_basket:
                    transactions_list.append({
                        'game_id': game_id,
                        'player_uuid': player_uuid,
                        'player_name': player_name,
                        'items': sorted(current_basket)
                    })
                # Reset inventory for their next life
                current_basket = set()
                
    # Add the final basket at the end of the game (if they still had items)
    if current_basket:
        transactions_list.append({
            'game_id': game_id,
            'player_uuid': player_uuid,
            'player_name': player_name,
            'items': sorted(current_basket)
        })

purchase_tx = pd.DataFrame(transactions_list)

print('Number of player-life purchase transactions:', len(purchase_tx))
purchase_tx.head(10)

Number of player-life purchase transactions: 104


,game_id,player_uuid,player_name,items
0,1,4028be1d-4312-4912-99c7-fef783b65945,Smart123s,"[ENCHANT:FallProt, ENCHANT:Unbreaking, WOOD_SWORD]"
1,1,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,[WOOD_SWORD]
2,1,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,"[STONE_SWORD, WOOD_SWORD]"
3,2,52a6c4d4-8f4a-3565-b3c6-3d2f57468391,Csani44114411,[GOLD_SWORD]
4,2,52a6c4d4-8f4a-3565-b3c6-3d2f57468391,Csani44114411,"[ENCHANT:Protection, ENCHANT:Unbreaking, GOLD_SWORD]"
5,2,52a6c4d4-8f4a-3565-b3c6-3d2f57468391,Csani44114411,"[ARROW, BOW, DIAMOND_SWORD, SNOW_BALL]"
6,2,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,"[DIAMOND_CHESTPLATE, GOLD_SWORD, WOOD_SWORD]"
7,2,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,"[ARROW, BOW, ENCHANT:Flame, GOLD_SWORD, STONE_SWORD]"
8,3,4028be1d-4312-4912-99c7-fef783b65945,Smart123s,[WOOD_SWORD]
9,3,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,[WOOD_SWORD]


## 5. Helpers: one-hot encoding, frequent itemsets, rules

In [64]:
def one_hot_from_transactions(transactions):
    te = TransactionEncoder()
    arr = te.fit(transactions).transform(transactions)
    return pd.DataFrame(arr, columns=te.columns_)


def frequent_itemsets_to_df(freq_dict, n_transactions: int) -> pd.DataFrame:
    rows = []
    for items, support_count in freq_dict.items():
        itemset = frozenset(items)
        rows.append({
            'itemsets': itemset,
            'support_count': int(support_count),
            'support': support_count / n_transactions,
        })
    if not rows:
        return pd.DataFrame(columns=['itemsets', 'support_count', 'support'])
    return pd.DataFrame(rows).sort_values(['support_count', 'support'], ascending=[False, False]).reset_index(drop=True)


def generate_rules_from_itemsets(freq_df: pd.DataFrame, min_conf: float = 0.6, single_consequent_only: bool = False, max_antecedent_size=None) -> pd.DataFrame:
    if freq_df.empty:
        return pd.DataFrame(columns=['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction'])

    support_map = {frozenset(items): supp for items, supp in zip(freq_df['itemsets'], freq_df['support'])}
    rows = []

    for itemset, supp_ab in support_map.items():
        if len(itemset) < 2:
            continue
        for r in range(1, len(itemset)):
            if max_antecedent_size is not None and r > max_antecedent_size:
                continue
            for antecedent_tuple in combinations(itemset, r):
                antecedent = frozenset(antecedent_tuple)
                consequent = itemset - antecedent
                if single_consequent_only and len(consequent) != 1:
                    continue

                supp_a = support_map.get(antecedent)
                supp_b = support_map.get(consequent)
                if not supp_a or not supp_b:
                    continue

                confidence = supp_ab / supp_a
                if confidence < min_conf:
                    continue

                lift = confidence / supp_b if supp_b else np.nan
                leverage = supp_ab - supp_a * supp_b
                conviction = (1 - supp_b) / (1 - confidence) if confidence < 1 else np.inf

                rows.append({
                    'antecedents': antecedent,
                    'consequents': consequent,
                    'support': supp_ab,
                    'antecedent support': supp_a,
                    'consequent support': supp_b,
                    'confidence': confidence,
                    'lift': lift,
                    'leverage': leverage,
                    'conviction': conviction,
                })

    if not rows:
        return pd.DataFrame(columns=['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction'])

    return pd.DataFrame(rows).sort_values(['lift', 'confidence', 'support'], ascending=[False, False, False]).reset_index(drop=True)


def itemset_to_str(itemset):
    return ', '.join(sorted(itemset))


def pretty_rules(rules: pd.DataFrame, cols=None):
    if rules.empty:
        return rules
    out = rules.copy()
    out['antecedents'] = out['antecedents'].apply(itemset_to_str)
    out['consequents'] = out['consequents'].apply(itemset_to_str)
    if cols is None:
        cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction']
    return out[cols]

## 6. Algorithm implementations

### Apriori
Implemented with `mlxtend.frequent_patterns.apriori`.

### AprioriTID
Custom implementation with transaction reduction after each level.

### ECLAT
Custom implementation using **vertical TID sets**.

### FP-Growth
Implemented with `mlxtend.frequent_patterns.fpgrowth`.


In [65]:
def aprioritid(transactions, min_support: float = 0.1) -> pd.DataFrame:
    transactions = [frozenset(t) for t in transactions]
    n = len(transactions)
    min_count = math.ceil(min_support * n)

    # L1
    item_counts = defaultdict(int)
    for tx in transactions:
        for item in tx:
            item_counts[frozenset([item])] += 1

    lk = {itemset: count for itemset, count in item_counts.items() if count >= min_count}
    all_freq = dict(lk)

    # reduced transaction representation
    ctk = []
    for tid, tx in enumerate(transactions):
        contained = {itemset for itemset in lk if itemset.issubset(tx)}
        if contained:
            ctk.append((tid, contained))

    k = 1
    while lk:
        lk_list = sorted(lk.keys(), key=lambda s: tuple(sorted(s)))
        ck1 = set()

        # candidate generation + Apriori pruning
        for i in range(len(lk_list)):
            for j in range(i + 1, len(lk_list)):
                left = sorted(lk_list[i])
                right = sorted(lk_list[j])
                if left[:-1] != right[:-1]:
                    break
                cand = frozenset(set(lk_list[i]) | set(lk_list[j]))
                if len(cand) == k + 1 and all(frozenset(sub) in lk for sub in combinations(cand, k)):
                    ck1.add(cand)

        if not ck1:
            break

        counts = defaultdict(int)
        ctk1 = []
        for tid, contained_k in ctk:
            items_in_tx = frozenset().union(*contained_k) if contained_k else frozenset()
            contained_next = set()
            for cand in ck1:
                if cand.issubset(items_in_tx):
                    contained_next.add(cand)
                    counts[cand] += 1
            if contained_next:
                ctk1.append((tid, contained_next))

        lk = {cand: count for cand, count in counts.items() if count >= min_count}
        if not lk:
            break

        all_freq.update(lk)
        ctk = []
        for tid, contained in ctk1:
            filtered = {cand for cand in contained if cand in lk}
            if filtered:
                ctk.append((tid, filtered))
        k += 1

    return frequent_itemsets_to_df(all_freq, n)


def eclat(transactions, min_support: float = 0.1) -> pd.DataFrame:
    transactions = [frozenset(t) for t in transactions]
    n = len(transactions)
    min_count = math.ceil(min_support * n)

    tidsets = defaultdict(set)
    for tid, tx in enumerate(transactions):
        for item in tx:
            tidsets[item].add(tid)

    items = sorted(
        [(frozenset([item]), tids) for item, tids in tidsets.items() if len(tids) >= min_count],
        key=lambda pair: tuple(sorted(pair[0])),
    )

    freq = {itemset: len(tids) for itemset, tids in items}

    def recurse(prefix, suffix_items):
        for i, (itemset_i, tids_i) in enumerate(suffix_items):
            next_suffix = []
            for itemset_j, tids_j in suffix_items[i + 1:]:
                candidate = prefix | itemset_i | itemset_j
                tids_c = tids_i & tids_j
                if len(tids_c) >= min_count:
                    freq[candidate] = len(tids_c)
                    next_suffix.append((itemset_j, tids_c))
            if next_suffix:
                recurse(prefix | itemset_i, next_suffix)

    recurse(frozenset(), items)
    return frequent_itemsets_to_df(freq, n)


def compare_algorithms(transactions, min_support: float = 0.12):
    results = {}
    timings = []

    one_hot = one_hot_from_transactions(transactions)

    start = time.perf_counter()
    apriori_df = apriori(one_hot, min_support=min_support, use_colnames=True)
    timings.append({'algorithm': 'Apriori', 'seconds': time.perf_counter() - start, 'itemsets': len(apriori_df)})
    apriori_df['support_count'] = (apriori_df['support'] * len(transactions)).round().astype(int)
    apriori_df = apriori_df[['itemsets', 'support_count', 'support']].sort_values(['support_count', 'support'], ascending=[False, False]).reset_index(drop=True)
    results['Apriori'] = apriori_df

    start = time.perf_counter()
    aprioritid_df = aprioritid(transactions, min_support=min_support)
    timings.append({'algorithm': 'AprioriTID', 'seconds': time.perf_counter() - start, 'itemsets': len(aprioritid_df)})
    results['AprioriTID'] = aprioritid_df

    start = time.perf_counter()
    eclat_df = eclat(transactions, min_support=min_support)
    timings.append({'algorithm': 'ECLAT', 'seconds': time.perf_counter() - start, 'itemsets': len(eclat_df)})
    results['ECLAT'] = eclat_df

    start = time.perf_counter()
    fpgrowth_df = fpgrowth(one_hot, min_support=min_support, use_colnames=True)
    timings.append({'algorithm': 'FP-Growth', 'seconds': time.perf_counter() - start, 'itemsets': len(fpgrowth_df)})
    fpgrowth_df['support_count'] = (fpgrowth_df['support'] * len(transactions)).round().astype(int)
    fpgrowth_df = fpgrowth_df[['itemsets', 'support_count', 'support']].sort_values(['support_count', 'support'], ascending=[False, False]).reset_index(drop=True)
    results['FP-Growth'] = fpgrowth_df

    canonical = None
    same_output = True
    for name, df in results.items():
        key = {tuple(sorted(itemset)) for itemset in df['itemsets']}
        if canonical is None:
            canonical = key
        else:
            same_output = same_output and (key == canonical)

    return results, pd.DataFrame(timings), same_output

## 7. Run all four algorithms on the purchase transactions

We start with a minimum support of **0.12**. With 24 transactions that means at least about **3** player-game baskets.


In [66]:
purchase_results, purchase_timing, purchase_same_output = compare_algorithms(
    purchase_tx['items'].tolist(),
    min_support=0.12,
)

print('All four algorithms found the same frequent itemsets:', purchase_same_output)
purchase_timing

All four algorithms found the same frequent itemsets: True


,algorithm,seconds,itemsets
0,Apriori,0.001479,5
1,AprioriTID,0.002487,5
2,ECLAT,0.003333,5
3,FP-Growth,0.002165,5


In [67]:
purchase_results['Apriori'].head(20)

,itemsets,support_count,support
0,frozenset({GOLD_SWORD}),39,0.375000
1,frozenset({WOOD_SWORD}),27,0.259615
2,frozenset({LEATHER_CHESTPLATE}),17,0.163462
3,frozenset({IRON_SWORD}),14,0.134615
4,frozenset({SNOW_BALL}),13,0.125000


## 8. Generate purchase rules

The slides define the standard two-step workflow:

1. find frequent itemsets
2. generate strong rules from them

Below we generate rules from the Apriori frequent itemsets. Since all four algorithms returned the same frequent itemsets above, the rule base is the same.


In [68]:
purchase_rules = generate_rules_from_itemsets(
    purchase_results['Apriori'],
    min_conf=0.60,
    single_consequent_only=True,
    max_antecedent_size=2,
)

pretty_rules(purchase_rules).head(25)

,antecedents,consequents,support,confidence,lift,leverage,conviction


In [69]:
# Example requested in the task: bow -> arrow
bow_arrow_rules = purchase_rules[
    purchase_rules['antecedents'].apply(lambda s: 'BOW' in s or 'ARROW' in s)
    & purchase_rules['consequents'].apply(lambda s: 'BOW' in s or 'ARROW' in s)
]

pretty_rules(bow_arrow_rules)

,antecedents,consequents,support,confidence,lift,leverage,conviction


In [70]:
# Helper: search for rules containing a token

def rules_containing(rules: pd.DataFrame, token: str):
    mask = rules['antecedents'].apply(lambda s: token in s) | rules['consequents'].apply(lambda s: token in s)
    return pretty_rules(rules.loc[mask])

rules_containing(purchase_rules, 'ENCHANT:Flame').head(20)

,antecedents,consequents,support,confidence,lift,leverage,conviction


## 9. Player-game summary for multidimensional associations

To mine **multidimensional** rules we enrich each player-game transaction with extra dimensions:

- player name
- game size
- best sword tier
- best armor tier
- total spend bucket
- kills bucket
- deaths bucket
- KD bucket
- bow / arrow / snowball usage
- win proxy

This lets us build rules such as:

- `BOW -> ARROW`
- `DIAMOND -> WIN_PROXY:YES`
- performance-related rules using purchases + deaths/kills together

For numeric attributes we use **static discretization** (`pd.cut`-style manual buckets), which is consistent with the slide deck discussion.


In [71]:
def build_player_game_summary(purchase: pd.DataFrame, death: pd.DataFrame, game_player: pd.DataFrame) -> pd.DataFrame:
    pg = pd.concat([
        game_player[['game_id', 'player_uuid', 'player_name']],
        purchase[['game_id', 'player_uuid', 'player_name']].drop_duplicates(),
        death.rename(columns={'murderer_uuid': 'player_uuid', 'murderer_name': 'player_name'})[['game_id', 'player_uuid', 'player_name']],
        death.rename(columns={'dead_uuid': 'player_uuid', 'dead_name': 'player_name'})[['game_id', 'player_uuid', 'player_name']],
    ], ignore_index=True).drop_duplicates()

    game_sizes = pg.groupby('game_id')['player_uuid'].nunique().rename('n_players').reset_index()
    kills = death.groupby(['game_id', 'murderer_uuid']).size().rename('kills').reset_index().rename(columns={'murderer_uuid': 'player_uuid'})
    deaths = death.groupby(['game_id', 'dead_uuid']).size().rename('deaths').reset_index().rename(columns={'dead_uuid': 'player_uuid'})
    m_reward = death.groupby(['game_id', 'murderer_uuid'])['murderer_reward'].sum().rename('murderer_reward_total').reset_index().rename(columns={'murderer_uuid': 'player_uuid'})
    d_reward = death.groupby(['game_id', 'dead_uuid'])['dead_reward'].sum().rename('dead_reward_total').reset_index().rename(columns={'dead_uuid': 'player_uuid'})

    purch_agg = purchase.groupby(['game_id', 'player_uuid']).agg(
        purchase_events=('id', 'count'),
        total_spend=('price', 'sum'),
        unique_items=('item', 'nunique'),
    ).reset_index()

    won = purchase.groupby(['game_id', 'player_uuid'])['item_base'].apply(lambda s: int((s == 'DIAMOND').any())).rename('won').reset_index()

    def indicator(base):
        flags = purchase.assign(flag=(purchase['item_base'] == base).astype(int))
        return flags.groupby(['game_id', 'player_uuid'])['flag'].max().rename(base.lower()).reset_index()

    bow = indicator('BOW')
    arrow = indicator('ARROW')
    snow = indicator('SNOW_BALL')

    sword_rank = {'WOOD_SWORD': 1, 'GOLD_SWORD': 2, 'STONE_SWORD': 3, 'IRON_SWORD': 4, 'DIAMOND_SWORD': 5}
    armor_rank = {'Leather': 1, 'Chain': 2, 'Gold': 2, 'Iron': 3, 'Diamond': 4}

    swords = purchase[purchase['item_base'].isin(sword_rank)].copy()
    if swords.empty:
        best_sword = pd.DataFrame(columns=['game_id', 'player_uuid', 'best_sword_rank', 'best_sword'])
    else:
        swords['rank'] = swords['item_base'].map(sword_rank)
        idx = swords.groupby(['game_id', 'player_uuid'])['rank'].idxmax()
        best_sword = swords.loc[idx, ['game_id', 'player_uuid', 'rank', 'item_base']].rename(columns={'rank': 'best_sword_rank', 'item_base': 'best_sword'})

    armors = purchase[purchase['main_category'] == 'Armor'].copy()
    if armors.empty:
        best_armor = pd.DataFrame(columns=['game_id', 'player_uuid', 'best_armor_rank', 'best_armor'])
    else:
        armors['rank'] = armors['sub_category'].map(armor_rank)
        idx = armors.groupby(['game_id', 'player_uuid'])['rank'].idxmax()
        best_armor = armors.loc[idx, ['game_id', 'player_uuid', 'rank', 'sub_category']].rename(columns={'rank': 'best_armor_rank', 'sub_category': 'best_armor'})

    summary = pg.merge(game_sizes, on='game_id', how='left')
    for df in [purch_agg, won, bow, arrow, snow, best_sword, best_armor, kills, deaths, m_reward, d_reward]:
        summary = summary.merge(df, on=['game_id', 'player_uuid'], how='left')

    zero_cols = ['purchase_events', 'total_spend', 'unique_items', 'won', 'bow', 'arrow', 'snow_ball', 'kills', 'deaths', 'murderer_reward_total', 'dead_reward_total']
    for col in zero_cols:
        summary[col] = summary[col].fillna(0)

    summary['best_sword_rank'] = summary['best_sword_rank'].fillna(0)
    summary['best_armor_rank'] = summary['best_armor_rank'].fillna(0)
    summary['best_sword'] = summary['best_sword'].fillna('NONE')
    summary['best_armor'] = summary['best_armor'].fillna('NONE')
    summary['kd_ratio'] = np.where(summary['deaths'] > 0, summary['kills'] / summary['deaths'], summary['kills'])
    summary['net_reward'] = summary['murderer_reward_total'] - summary['dead_reward_total']

    return summary.sort_values(['game_id', 'player_name']).reset_index(drop=True)


summary = build_player_game_summary(purchase, death_raw, game_player)
summary.head(20)

,game_id,player_uuid,player_name,n_players,purchase_events,total_spend,unique_items,won,bow,arrow,...,best_sword_rank,best_sword,best_armor_rank,best_armor,kills,deaths,murderer_reward_total,dead_reward_total,kd_ratio,net_reward
0,1,4028be1d-4312-4912-99c7-fef783b65945,Smart123s,2,3.0,350.0,3.0,0.0,0.0,0.0,...,1.0,WOOD_SWORD,0.0,NONE,1.0,1.0,226.0,662.0,1.000000,-436.0
1,1,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,2,3.0,800.0,2.0,0.0,0.0,0.0,...,3.0,STONE_SWORD,0.0,NONE,1.0,1.0,674.0,250.0,1.000000,424.0
2,2,52a6c4d4-8f4a-3565-b3c6-3d2f57468391,Csani44114411,2,23.0,3926.0,7.0,0.0,1.0,1.0,...,5.0,DIAMOND_SWORD,0.0,NONE,2.0,2.0,4031.0,6171.0,1.000000,-2140.0
3,2,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,2,28.0,6250.0,7.0,0.0,1.0,1.0,...,3.0,STONE_SWORD,4.0,Diamond,2.0,2.0,5748.0,4390.0,1.000000,1358.0
4,3,4028be1d-4312-4912-99c7-fef783b65945,Smart123s,2,1.0,200.0,1.0,0.0,0.0,0.0,...,1.0,WOOD_SWORD,0.0,NONE,0.0,0.0,0.0,0.0,0.000000,0.0
5,3,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,2,1.0,200.0,1.0,0.0,0.0,0.0,...,1.0,WOOD_SWORD,0.0,NONE,0.0,0.0,0.0,0.0,0.000000,0.0
6,4,4028be1d-4312-4912-99c7-fef783b65945,Smart123s,2,1.0,200.0,1.0,0.0,0.0,0.0,...,1.0,WOOD_SWORD,0.0,NONE,0.0,1.0,0.0,250.0,0.000000,-250.0
7,4,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,2,1.0,200.0,1.0,0.0,0.0,0.0,...,1.0,WOOD_SWORD,0.0,NONE,1.0,0.0,226.0,0.0,1.000000,226.0
8,5,4028be1d-4312-4912-99c7-fef783b65945,Smart123s,2,1.0,200.0,1.0,0.0,0.0,0.0,...,1.0,WOOD_SWORD,0.0,NONE,0.0,1.0,0.0,250.0,0.000000,-250.0
9,5,c83992bb-839f-4a9e-a164-935566835e4e,Smart123ss,2,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NONE,0.0,NONE,1.0,0.0,226.0,0.0,1.000000,226.0


In [ ]:
# Show win-proxy candidates and the anomaly mentioned in the task
purchase.loc[purchase['item_base'] == 'DIAMOND', ['game_id', 'player_name', 'price', 'created_at']]    .sort_values(['game_id', 'player_name'])

In [ ]:
def bin_kills(x):
    if x == 0:
        return 'none'
    if x <= 2:
        return 'low'
    if x <= 6:
        return 'medium'
    return 'high'


def bin_deaths(x):
    if x == 0:
        return 'none'
    if x <= 2:
        return 'low'
    if x <= 6:
        return 'medium'
    return 'high'


def bin_spend(x):
    if x == 0:
        return 'zero'
    if x <= 500:
        return 'low'
    if x <= 2500:
        return 'medium'
    if x <= 10000:
        return 'high'
    return 'very_high'


def bin_kd(x):
    if x == 0:
        return 'zero'
    if x < 0.75:
        return 'low'
    if x <= 1.25:
        return 'even'
    return 'high'


def bin_purchase_events(x):
    if x == 0:
        return 'none'
    if x <= 3:
        return 'few'
    if x <= 10:
        return 'many'
    return 'spam'


def build_multidim_transactions(summary: pd.DataFrame, purchase: pd.DataFrame, include_player: bool = False) -> pd.DataFrame:
    purchase_sets = purchase.groupby(['game_id', 'player_uuid'])['canonical_token'].agg(lambda s: sorted(set(s))).rename('purchase_items').reset_index()
    multi = summary.merge(purchase_sets, on=['game_id', 'player_uuid'], how='left')
    multi['purchase_items'] = multi['purchase_items'].apply(lambda x: x if isinstance(x, list) else [])

    multi['kills_bin'] = multi['kills'].map(bin_kills)
    multi['deaths_bin'] = multi['deaths'].map(bin_deaths)
    multi['spend_bin'] = multi['total_spend'].map(bin_spend)
    multi['kd_bin'] = multi['kd_ratio'].map(bin_kd)
    multi['purchase_events_bin'] = multi['purchase_events'].map(bin_purchase_events)

    def row_items(row):
        items = set(row['purchase_items'])
        items.add(f"GAME_SIZE:{int(row['n_players'])}")
        items.add(f"SWORD:{row['best_sword']}")
        items.add(f"ARMOR:{row['best_armor']}")
        items.add(f"KILLS:{row['kills_bin']}")
        items.add(f"DEATHS:{row['deaths_bin']}")
        items.add(f"KD:{row['kd_bin']}")
        items.add(f"SPEND:{row['spend_bin']}")
        items.add(f"PURCHASES:{row['purchase_events_bin']}")
        items.add('WIN_PROXY:YES' if row['won'] > 0 else 'WIN_PROXY:NO')
        if row['bow'] > 0:
            items.add('USES_BOW')
        if row['arrow'] > 0:
            items.add('USES_ARROW')
        if row['snow_ball'] > 0:
            items.add('USES_SNOWBALL')
        if include_player:
            items.add(f"PLAYER:{row['player_name']}")
        return sorted(items)

    multi['items'] = multi.apply(row_items, axis=1)
    return multi


multi = build_multidim_transactions(summary, purchase, include_player=False)
print('Multidimensional transactions:', len(multi))
multi[['game_id', 'player_name', 'items']].head(10)

## 10. Mine multidimensional associations

To keep the rule space readable, we cap frequent itemsets at **length 3** here.
This keeps the notebook responsive and still gives useful hybrid rules.


In [ ]:
min_support_multi = 2 / len(multi)  # allow rules that occur at least twice
one_hot_multi = one_hot_from_transactions(multi['items'].tolist())

fi_multi = apriori(
    one_hot_multi,
    min_support=min_support_multi,
    use_colnames=True,
    max_len=3,
)
fi_multi['support_count'] = (fi_multi['support'] * len(multi)).round().astype(int)
fi_multi = fi_multi[['itemsets', 'support_count', 'support']].sort_values(['support_count', 'support'], ascending=[False, False]).reset_index(drop=True)

fi_multi.head(25)

In [ ]:
multi_rules = generate_rules_from_itemsets(
    fi_multi,
    min_conf=0.60,
    single_consequent_only=True,
    max_antecedent_size=2,
)

pretty_rules(multi_rules).head(25)

In [ ]:
def filter_rules_with_consequents(rules: pd.DataFrame, consequents):
    consequents = set(consequents)
    mask = rules['consequents'].apply(lambda s: len(s) == 1 and next(iter(s)) in consequents)
    return rules.loc[mask].reset_index(drop=True)


interesting_multi = filter_rules_with_consequents(
    multi_rules,
    {'WIN_PROXY:YES', 'DEATHS:high', 'DEATHS:medium', 'KILLS:high', 'KILLS:medium'}
)

pretty_rules(interesting_multi).head(40)

In [ ]:
# Strongest win-proxy rules
win_rules = filter_rules_with_consequents(multi_rules, {'WIN_PROXY:YES'})
pretty_rules(win_rules).head(20)

In [ ]:
# Search for rules involving sword choices
pretty_rules(multi_rules[
    multi_rules['antecedents'].apply(lambda s: 'WOOD_SWORD' in s or 'DIAMOND_SWORD' in s or 'IRON_SWORD' in s)
]).head(40)

## 11. Interestingness measures

The uploaded slides explicitly use these measures:

- **Support**
- **Confidence**
- **Lift**

The notebook computes them directly for each rule:

- `support(A -> B) = P(A ∪ B)`
- `confidence(A -> B) = P(B | A) = P(A ∪ B) / P(A)`
- `lift(A -> B) = P(A ∪ B) / (P(A) P(B))`

Interpretation of lift:

- `lift = 1` -> independence
- `lift > 1` -> positive association
- `lift < 1` -> negative association

For convenience, the notebook also reports:

- **leverage**
- **conviction**

Those two are not required by the slide formulas, but they are often useful in practice.


In [ ]:
# Compact final views
purchase_rules_view = pretty_rules(purchase_rules, ['antecedents', 'consequents', 'support', 'confidence', 'lift'])
multi_rules_view = pretty_rules(interesting_multi, ['antecedents', 'consequents', 'support', 'confidence', 'lift'])

print('Top purchase rules:')
display(purchase_rules_view.head(20))

print('Top multidimensional outcome-related rules:')
display(multi_rules_view.head(20))

## 12. Optional extension ideas

If you want to extend the assignment further, these are the easiest next steps:

1. include `PLAYER:<name>` in the multidimensional transactions
2. try a lower / higher minimum support threshold
3. compare category-level rules such as `Sword -> Armor` or `Bow -> Arrow`
4. replace manual buckets with quantile-based discretization
5. add charting for support / confidence / lift distributions
